### Simple Markov Chain Text Generator

A Markov chain works by mapping a 'state' (like a word) to a set of possible 'next states' (the words that followed it in the source text).

1. **Training**: We iterate through the text, and for every word, we record what word came next.
2. **Generation**: We start with a random word and 'hop' to a next word based on the frequencies observed during training.

In [1]:
import random
from collections import defaultdict

class MarkovChain:
    def __init__(self):
        # Use a dictionary of lists to store possible transitions
        self.chain = defaultdict(list)

    def train(self, text):
        words = text.split()
        for i in range(len(words) - 1):
            current_word = words[i]
            next_word = words[i + 1]
            self.chain[current_word].append(next_word)

    def generate(self, start_word=None, length=20):
        if not self.chain:
            return "Model not trained."

        if start_word is None or start_word not in self.chain:
            start_word = random.choice(list(self.chain.keys()))

        result = [start_word]
        current_word = start_word

        for _ in range(length - 1):
            next_words = self.chain.get(current_word)
            if not next_words:
                break
            current_word = random.choice(next_words)
            result.append(current_word)

        return " ".join(result)

# Sample text (a bit of Lorem Ipsum for variety)
sample_text = """
In the beginning God created the heaven and the earth.
And the earth was without form, and void; and darkness was upon the face of the deep.
And the Spirit of God moved upon the face of the waters.
And God said, Let there be light: and there was light.
"""

# Initialize and train
model = MarkovChain()
model.train(sample_text)

# Generate some text
generated_text = model.generate(length=15)
print(f"Generated Text:\n{generated_text}")

Generated Text:
and the face of God said, Let there be light: and void; and there was


In [2]:
!pip install nltk
!pip install spacy
!pip install markovify
!pip install -m spacy download en

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 6.3 MB/s eta 0:00:00

Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -m


In [3]:
import spacy
import re
import markovify
import nltk
from nltk.corpus import gutenberg
import warnings
warnings.filterwarnings('ignore')

In [8]:
nltk.download('gutenberg')
!python -m spacy download en

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 91.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [12]:
#inspect Gutenberg corpus
print(gutenberg.fileids())



['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']


In [13]:
#import novels as text objects
hamlet = gutenberg.raw('shakespeare-hamlet.txt')
macbeth = gutenberg.raw('shakespeare-macbeth.txt')
caesar = gutenberg.raw('shakespeare-caesar.txt')
#print first 100 characters of each
print('\nRaw:\n', hamlet[:100])
print('\nRaw:\n', macbeth[:100])
print('\nRaw:\n', caesar[:100])


Raw:
 [The Tragedie of Hamlet by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Barnardo a

Raw:
 [The Tragedie of Macbeth by William Shakespeare 1603]


Actus Primus. Scoena Prima.

Thunder and Lig

Raw:
 [The Tragedie of Julius Caesar by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Fla


In [14]:
#utility function for text cleaning
def text_cleaner(text):
  text = re.sub(r'--', ' ', text)
  text = re.sub('[[].*?[]]', '', text)
  text = re.sub(r'(b|s+-?|^-?)(d+|d*.d+)b','', text)
  text = ' '.join(text.split())
  return text

In [15]:
#remove chapter indicator
hamlet = re.sub(r'Chapter d+', '', hamlet)
macbeth = re.sub(r'Chapter d+', '', macbeth)
caesar = re.sub(r'Chapter d+', '', caesar)

In [16]:
#apply cleaning function to corpus
hamlet = text_cleaner(hamlet)
caesar = text_cleaner(caesar)
macbeth = text_cleaner(macbeth)

In [28]:
hamlet_sents = ' '.join([sent.text for sent in hamlet_doc.sents if len(sent.text) > 1])
macbeth_sents = ' '.join([sent.text for sent in macbeth_doc.sents if len(sent.text) > 1])
caesar_sents = ' '.join([sent.text for sent in caesar_doc.sents if len(sent.text)>1])

## Fix Spacy Loading and Parse Texts

### Subtask:
Correct the spaCy model loading and parse the cleaned Shakespearean texts into spaCy Doc objects.


In [18]:
import spacy

# Correctly load the spaCy model using the full name
nlp = spacy.load('en_core_web_sm')

# Parse the cleaned texts into spaCy Doc objects
hamlet_doc = nlp(hamlet)
macbeth_doc = nlp(macbeth)
caesar_doc = nlp(caesar)

print("Successfully parsed hamlet, macbeth, and caesar into spaCy Doc objects.")

Successfully parsed hamlet, macbeth, and caesar into spaCy Doc objects.


## Combine and Train Manual Model

### Subtask:
Combine the cleaned Shakespearean texts into a single corpus and train the custom MarkovChain model.


In [21]:
combined_text = hamlet + ' ' + macbeth + ' ' + caesar

manual_model = MarkovChain()
manual_model.train(combined_text)

print(f'Total unique words in the Markov chain: {len(manual_model.chain)}')
print('Manual MarkovChain model trained successfully.')

Total unique words in the Markov chain: 13393
Manual MarkovChain model trained successfully.


## Implement Markovify Model

### Subtask:
Initialize and train a `markovify.Text` model on the combined Shakespearean corpus for sentence-level generation.


In [22]:
import markovify

# Initialize the markovify model using the combined text of the three plays
# state_size=2 is a good balance between coherence and variety
markovify_model = markovify.Text(combined_text, state_size=2)

print("markovify.Text model initialized and trained successfully.")

markovify.Text model initialized and trained successfully.


## Generate and Compare Text Samples

### Subtask:
Generate text samples using both the custom `MarkovChain` model and the `markovify.Text` model, then compare their output to evaluate coherence and style.

In [23]:
# Generate a sample from the manual MarkovChain model
# We specify a length of 40 words to get a substantial sample
manual_sample = manual_model.generate(length=40)

# Generate a sample sentence from the markovify model
# make_sentence() attempts to create a coherent sentence based on the overlapping states
markovify_sample = markovify_model.make_sentence(tries=100)

print("--- Custom MarkovChain Generated Text ---")
print(manual_sample)

print("\n--- Markovify Generated Sentence ---")
if markovify_sample:
    print(markovify_sample)
else:
    print("Markovify was unable to generate a valid sentence with the current constraints.")

--- Custom MarkovChain Generated Text ---
Rood, not giue you shall raise no farther Bru. Go make inquiry Of Honorable Mettle should applaud the Polake warres, and still hang aliue Till Famine cling thee: Naught that Friend should be as pure as a Torch. 2. Ile

--- Markovify Generated Sentence ---
Then hee offered it to our hope.


In [29]:
# Using the previously defined markovify_model as generator_1
generator_1 = markovify_model

print("--- Standard Sentences ---")
for i in range(3):
  print(generator_1.make_sentence())

print("\n--- Short Sentences (max 100 chars) ---")
for i in range(3):
  print(generator_1.make_short_sentence(max_chars=100))

--- Standard Sentences ---
Madam not yet, I go to bed Ham.
I am not well vnderstand that.
Was not that I may deliuer Vpon the foot of mine eye.

--- Short Sentences (max 100 chars) ---
O my Lord Ham.
What's the Boy heard me: Brutus hath riu'd my hart: A Friend should beare his Comment Bru.
Harke, harke, one knockes: Portia go to do Malc.


In [30]:
# Combine the sentences from all three plays
shakespeare_sents = hamlet_sents + ' ' + macbeth_sents + ' ' + caesar_sents

# next we will use spacy's part of speech to generate more legible text
class POSifiedText(markovify.Text):
   def word_split(self, sentence):
      return ['::'.join((word.orth_, word.pos_)) for word in nlp(sentence)]
   def word_join(self, words):
      sentence = ' '.join(word.split('::')[0] for word in words)
      return sentence

# Call the class on our combined text
generator_2 = POSifiedText(shakespeare_sents, state_size=3)

In [31]:
#now we will use the above generator to generate sentences
for i in range(5):
  print(generator_2.make_sentence())
#print 100 characters or less sentences
for i in range(5):
  print(generator_2.make_short_sentence(max_chars=100))

Perhaps he loues you now , receiue them Ham .
Go some of ye , And bring me their opinions of Successe Ser .
There 's a Daysie , I would know that Polon .
He had none  His flight was madnesse  Was't Hamlet wrong'd Laertes ?
As kill a King , Vpon whose property , and most deere life , A Sable Siluer'd Ham .
Not a iot more , my Lord , what haue I seene to night ?
Hamlet , thou hast bin all this while in a most fast sleepe Doct .
You can not speake of Reason to the Dane , And loose the name of Action .
This Caesar was a Tyrant 3 Nay that 's certaine  We are yet but yong indeed .
Speak no more of her  Giue me some Light .
